# Swin-T + Faster R-CNN · VisDrone · End-to-End Fine-tune
**Backbone + TransformerHead (Option 3) · Image 1024 · Batch 16**

Variation 1 — all components train jointly with differential LR:
Swin-T body `lr=5e-5` | FPN + RPN + head `lr=5e-4`

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # synchronous CUDA errors

!pip install -q pycocotools torchmetrics
from google.colab import drive
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 35.5 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import urllib.request
import zipfile
from pathlib import Path
from PIL import Image

DOWNLOAD_ROOT = Path.home() / 'visdrone' / 'data'
SPLITS = [
    'VisDrone2019-DET-train',
    'VisDrone2019-DET-val',
    'VisDrone2019-DET-test-dev',
]
URLS = {
    'VisDrone2019-DET-train':
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
    'VisDrone2019-DET-val':
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
    'VisDrone2019-DET-test-dev':
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip',
}

def _progress(b, bs, total):
    if total > 0:
        print(f'\r  {min(b*bs/total*100,100):.1f}%  {b*bs//1_048_576} / {total//1_048_576} MB',
              end='', flush=True)

def download_split(split):
    d = DOWNLOAD_ROOT / split
    if d.exists() and any(d.iterdir()):
        print(f'  {split}: already present, skipping')
        return
    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    zp = DOWNLOAD_ROOT / f'{split}.zip'
    print(f'  downloading {split} ...')
    urllib.request.urlretrieve(URLS[split], zp, reporthook=_progress)
    print()
    with zipfile.ZipFile(zp,'r') as z: z.extractall(DOWNLOAD_ROOT)
    zp.unlink()
    print(f'  extracted → {d}')

def convert_split(split_dir):
    ann_dir = split_dir / 'annotations'
    img_dir = split_dir / 'images'
    lbl_dir = split_dir / 'labels'
    if not ann_dir.exists(): return
    if lbl_dir.exists() and any(lbl_dir.glob('*.txt')):
        print(f'  {split_dir.name}: labels exist, skipping')
        return
    lbl_dir.mkdir(parents=True, exist_ok=True)
    skipped = 0
    for ap in sorted(ann_dir.glob('*.txt')):
        ip = (img_dir / ap.name).with_suffix('.jpg')
        try: iw,ih = Image.open(ip).size
        except: skipped+=1; continue
        lines=[]
        for row in open(ap).read().strip().splitlines():
            r=row.strip().split(',')
            if len(r)<6: continue
            cat=int(r[5])
            if cat==0: continue
            x,y,w,h=int(r[0]),int(r[1]),int(r[2]),int(r[3])
            if w<=0 or h<=0: continue
            lines.append(f'{cat-1} {(x+w/2)/iw:.6f} {(y+h/2)/ih:.6f} {w/iw:.6f} {h/ih:.6f}\n')
        open(lbl_dir/ap.name,'w').writelines(lines)
    print(f'  {split_dir.name}: converted  (skipped={skipped})')

for s in SPLITS:
    download_split(s)
    convert_split(DOWNLOAD_ROOT / s)

print('\nDataset structure:')
for s in SPLITS:
    ni = len(list((DOWNLOAD_ROOT/s/'images').glob('*.jpg')))
    print(f'  {s}: {ni:,} images')


  downloading VisDrone2019-DET-train ...
  100.0%  1478 / 1478 MB
  extracted → /root/visdrone/data/VisDrone2019-DET-train
  VisDrone2019-DET-train: converted  (skipped=0)
  downloading VisDrone2019-DET-val ...
  100.0%  77 / 77 MB
  extracted → /root/visdrone/data/VisDrone2019-DET-val
  VisDrone2019-DET-val: converted  (skipped=0)
  downloading VisDrone2019-DET-test-dev ...
  100.0%  296 / 296 MB
  extracted → /root/visdrone/data/VisDrone2019-DET-test-dev
  VisDrone2019-DET-test-dev: converted  (skipped=0)

Dataset structure:
  VisDrone2019-DET-train: 6,471 images
  VisDrone2019-DET-val: 548 images
  VisDrone2019-DET-test-dev: 1,610 images


In [3]:
import os, sys, math, time, json, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models import swin_t, Swin_T_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.ops import FeaturePyramidNetwork, MultiScaleRoIAlign, box_iou
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, precision_score, recall_score)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → A100'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.0f} GB')
print(f'Torch: {torch.__version__}  Torchvision: {torchvision.__version__}')
torch.backends.cudnn.benchmark = True


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  VRAM: 102 GB
Torch: 2.10.0+cu128  Torchvision: 0.25.0+cu128


In [24]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    scheduler = WarmupCosine(optimizer, WARMUP_STEPS, total_steps)

In [25]:
CLASS_NAMES = [
    'pedestrian','people','bicycle','car','van',
    'truck','tricycle','awning-tricycle','bus','motor',
]
N_CLS        = 10
N_CLS_WITH_BG= 11
VALID_CATS   = set(range(1, 11))   # categories 1-10, skip 0 (ignored)

IMG_MAX_SIZE = 1024
BATCH_SIZE   = 24
EPOCHS       = 30
LR_BACKBONE  = 5e-5    # Swin-T body  — conservative (pretrained)
LR_HEAD      = 5e-4    # FPN+RPN+head — same ratio as ResNet notebook (1:10)
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 200
NUM_WORKERS  = 8
TAG          = 'swint_opt3_e2e'

DATA_ROOT  = Path.home() / 'visdrone' / 'data'
CKPT_DIR   = Path('/content/drive/MyDrive/VisDrone/checkpoints') / TAG
PLOT_DIR   = Path('/content/drive/MyDrive/VisDrone/plots')       / TAG
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DIRS = {
    'train': 'VisDrone2019-DET-train',
    'val':   'VisDrone2019-DET-val',
    'test':  'VisDrone2019-DET-test-dev',
}
print(f'TAG     : {TAG}')
print(f'Ckpts   → {CKPT_DIR}')
print(f'Plots   → {PLOT_DIR}')


TAG     : swint_opt3_e2e
Ckpts   → /content/drive/MyDrive/VisDrone/checkpoints/swint_opt3_e2e
Plots   → /content/drive/MyDrive/VisDrone/plots/swint_opt3_e2e


In [26]:
class VisDroneDataset(Dataset):
    """
    VisDrone-DET dataset.  Reads original comma-separated annotations.
    Category 0 (ignored) skipped; categories 1-10 kept.
    Images resized to max_size on longest side (no letterbox padding —
    matches ResNet notebook behaviour for fair comparison).
    """
    def __init__(self, root, max_size=1024, augment=False, max_samples=None):
        self.ann_dir  = Path(root) / 'annotations'
        self.img_dir  = Path(root) / 'images'
        self.max_size = max_size
        self.augment  = augment
        if not self.img_dir.exists():
            raise FileNotFoundError(
                f'Image dir not found: {self.img_dir}\nRun download cell first.')
        self.img_files = sorted(
            f for f in os.listdir(str(self.img_dir))
            if f.lower().endswith(('.jpg','.jpeg','.png')))
        if not self.img_files:
            raise ValueError(f'No images in {self.img_dir}')
        if max_samples: self.img_files = self.img_files[:max_samples]
        print(f'  {len(self.img_files):,} images from {Path(root).name}')

    def __len__(self): return len(self.img_files)

    def __getitem__(self, idx):
        img_f = self.img_files[idx]
        img   = Image.open(self.img_dir / img_f).convert('RGB')
        ow,oh = img.size
        scale = min(self.max_size/ow, self.max_size/oh, 1.0)
        nw,nh = int(ow*scale), int(oh*scale)
        img   = img.resize((nw,nh), Image.BILINEAR)

        ap = self.ann_dir / (Path(img_f).stem + '.txt')
        boxes,labels = [],[]
        if ap.exists():
            for line in open(ap):
                p = line.strip().split(',')
                if len(p)<6: continue
                x,y,w,h = float(p[0]),float(p[1]),float(p[2]),float(p[3])
                cat = int(p[5])
                if cat not in VALID_CATS or w<2 or h<2: continue
                boxes.append([x*scale, y*scale,
                               (x+w)*scale, (y+h)*scale])
                labels.append(cat)

        img_t = TF.to_tensor(img)
        if not boxes:
            tgt = {'boxes':  torch.zeros((0,4), dtype=torch.float32),
                   'labels': torch.zeros((0,),  dtype=torch.int64),
                   'image_id': torch.tensor([idx])}
        else:
            bt = torch.tensor(boxes, dtype=torch.float32)
            bt = bt.clamp(min=0)
            bt[:,2] = bt[:,2].clamp(max=nw)
            bt[:,3] = bt[:,3].clamp(max=nh)
            keep = (bt[:,2]-bt[:,0]>1) & (bt[:,3]-bt[:,1]>1)
            tgt = {'boxes':  bt[keep],
                   'labels': torch.tensor(labels,dtype=torch.int64)[keep],
                   'image_id': torch.tensor([idx])}
        return img_t, tgt


def collate_fn(batch): return tuple(zip(*batch))


print('Loading datasets...')
train_ds = VisDroneDataset(DATA_ROOT/SPLIT_DIRS['train'], IMG_MAX_SIZE, augment=True)
val_ds   = VisDroneDataset(DATA_ROOT/SPLIT_DIRS['val'],   IMG_MAX_SIZE)
test_ds  = VisDroneDataset(DATA_ROOT/SPLIT_DIRS['test'],  IMG_MAX_SIZE)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, 2, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn,
                          pin_memory=True)
test_loader  = DataLoader(test_ds, 2, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn,
                          pin_memory=True)


Loading datasets...
  6,471 images from VisDrone2019-DET-train
  548 images from VisDrone2019-DET-val
  1,610 images from VisDrone2019-DET-test-dev


In [27]:
class SwinBackboneWithFPN(nn.Module):
    """
    Swin-T (ImageNet-1K) + 4-level FPN → 256-ch pyramid.
    Swin stages output channel-last [B,H,W,C]; permuted for FPN.
    At 1024px: s1=256×256, s2=128×128, s3=64×64, s4=32×32.
    """
    def __init__(self, out_channels=256, pretrained=True):
        super().__init__()
        swin = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
        self.body = create_feature_extractor(swin, return_nodes={
            'features.1':'s1','features.3':'s2',
            'features.5':'s3','features.7':'s4'})
        self.fpn  = FeaturePyramidNetwork(
            in_channels_list=[96,192,384,768], out_channels=out_channels)
        self.out_channels = out_channels

    def forward(self, x):
        f = self.body(x)
        # Swin is channel-last → permute to channel-first for FPN
        return self.fpn({k:v.permute(0,3,1,2).contiguous() for k,v in f.items()})


In [28]:
class TransformerBoxHead(nn.Module):
    """
    Option 3 RoI head — full self-attention over 7×7 RoI tokens.

    Input  [N, 256, 7, 7]  →  Output [N, 256]

    Architecture:
        flatten → [N,49,256] + pos_embed
        → pre-norm MHSA (4 heads, dim=256) + residual
        → pre-norm FFN  (256→512→256, GELU) + residual
        → mean pool → [N, 256]

    Parameter count:
        pos_embed  :   12,544
        MHSA block :  263,168
        FFN block  :  262,912
        ──────────────────────
        Sub-total  :  538,624   (~539K)
        Predictor  :   14,135   (FC cls + FC reg)
        ──────────────────────
        Total      :  552,759   (≈553K)
    """
    def __init__(self, in_channels=256, feat_dim=256, ffn_dim=512,
                 num_heads=4, roi_size=7, dropout=0.0):
        super().__init__()
        self.out_channels = feat_dim
        n_tok = roi_size**2
        self.proj      = (nn.Linear(in_channels,feat_dim)
                          if in_channels!=feat_dim else nn.Identity())
        self.pos_embed = nn.Parameter(torch.zeros(1,n_tok,feat_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.norm1 = nn.LayerNorm(feat_dim)
        self.attn  = nn.MultiheadAttention(feat_dim, num_heads,
                                            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(feat_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(feat_dim,ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim,feat_dim), nn.Dropout(dropout))

    def forward(self, x):
        t = x.flatten(2).transpose(1,2)           # [N,49,C]
        t = self.proj(t) + self.pos_embed
        n = self.norm1(t); a,_ = self.attn(n,n,n); t = t+a
        t = t + self.ffn(self.norm2(t))
        return t.mean(1)                           # [N,256]


In [29]:
def build_model(pretrained=True):
    """
    End-to-end variation: Swin-T + FPN + RPN + TransformerBoxHead.
    All parameters train. Differential LR applied in the optimizer.
    """
    bb   = SwinBackboneWithFPN(pretrained=pretrained)
    rpn  = AnchorGenerator(
        sizes        = ((32,),(64,),(128,),(256,)),
        aspect_ratios= ((0.5,1.0,2.0),)*4)
    pool = MultiScaleRoIAlign(['s1','s2','s3','s4'],
                               output_size=7, sampling_ratio=2)
    head = TransformerBoxHead()
    pred = FastRCNNPredictor(head.out_channels, N_CLS_WITH_BG)

    model = FasterRCNN(
        backbone=bb, num_classes=None,
        rpn_anchor_generator=rpn, box_roi_pool=pool,
        box_head=head, box_predictor=pred,
        rpn_pre_nms_top_n_train=3000, rpn_post_nms_top_n_train=2000,
        rpn_pre_nms_top_n_test=1500,  rpn_post_nms_top_n_test=700,
        box_score_thresh=0.05, box_nms_thresh=0.50,
        box_detections_per_img=300,
        min_size=IMG_MAX_SIZE, max_size=IMG_MAX_SIZE)

    n_total = sum(p.numel() for p in model.parameters())
    n_head  = sum(p.numel() for p in head.parameters())
    n_pred  = sum(p.numel() for p in pred.parameters())
    n_body  = sum(p.numel() for p in bb.body.parameters())
    n_fpn   = sum(p.numel() for p in bb.fpn.parameters())
    n_rpn   = sum(p.numel() for p in model.rpn.parameters())
    print(f'  Swin-T body    : {n_body/1e6:.2f}M  [lr={LR_BACKBONE:.0e}]')
    print(f'  FPN            : {n_fpn/1e6:.2f}M  [lr={LR_HEAD:.0e}]')
    print(f'  RPN            : {n_rpn/1e6:.2f}M  [lr={LR_HEAD:.0e}]')
    print(f'  TransformerHead: {(n_head+n_pred)/1e3:.0f}K  [lr={LR_HEAD:.0e}]')
    print(f'  Total          : {n_total/1e6:.2f}M  (all trainable)')
    return model


model = build_model(pretrained=True).to(device)


  Swin-T body    : 27.52M  [lr=5e-05]
  FPN            : 2.73M  [lr=5e-04]
  RPN            : 0.59M  [lr=5e-04]
  TransformerHead: 554K  [lr=5e-04]
  Total          : 31.40M  (all trainable)


In [30]:
# Differential LR: Swin-T body gets 10× lower LR than head
# — matches ResNet notebook ratio (backbone 5e-5, head 5e-4)
bb_ids  = {id(p) for p in model.backbone.body.parameters()}
bb_p    = [p for p in model.parameters() if id(p) in bb_ids]
head_p  = [p for p in model.parameters() if id(p) not in bb_ids]

optimizer = optim.AdamW([
    {'params': bb_p,   'lr': LR_BACKBONE},
    {'params': head_p, 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)


class WarmupCosine(optim.lr_scheduler._LRScheduler):
    """Linear warmup then cosine decay, applied per-step."""
    def __init__(self, opt, warmup, total, eta_min_r=0.05):
        self.warmup  = warmup
        self.total   = total
        self.eta_min = eta_min_r
        super().__init__(opt, last_epoch=-1)
    def get_lr(self):
        s = self.last_epoch
        return [
            base * ((s+1)/self.warmup if s<self.warmup else
                    self.eta_min + (1-self.eta_min)*0.5*(
                        1+math.cos(math.pi*(s-self.warmup)/
                                   max(1,self.total-self.warmup))))
            for base in self.base_lrs]


total_steps = len(train_loader) * EPOCHS
scheduler   = WarmupCosine(optimizer, WARMUP_STEPS, total_steps)
scaler      = GradScaler()

print(f'Optimizer  : AdamW')
print(f'LR backbone: {LR_BACKBONE:.0e}  |  LR head: {LR_HEAD:.0e}')
print(f'Epochs     : {EPOCHS}  |  Steps: {total_steps:,}  |  Warmup: {WARMUP_STEPS}')


Optimizer  : AdamW
LR backbone: 5e-05  |  LR head: 5e-04
Epochs     : 30  |  Steps: 8,070  |  Warmup: 200


/tmp/ipykernel_2231/371139944.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler      = GradScaler()


In [31]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    run_loss = 0.0
    comp_sum = defaultdict(float)
    nb = 0
    t0 = time.time()
    for bi,(images,targets) in enumerate(loader):
        images  = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]
        # skip batches with no boxes (prevents empty loss_dict)
        valid = [(i,t) for i,t in zip(images,targets) if t['boxes'].shape[0]>0]
        if not valid: continue
        images,targets = zip(*valid)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            ld   = model(list(images), list(targets))
            loss = sum(ld.values())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()

        run_loss += loss.item()
        for k,v in ld.items(): comp_sum[k] += v.item()
        nb += 1
        if bi%50==0:
            lr_bb = optimizer.param_groups[0]['lr']
            lr_hd = optimizer.param_groups[1]['lr']
            print(f'  [{epoch}][{bi:04d}/{len(loader):04d}]  '
                  f'loss={run_loss/max(nb,1):.4f}  '
                  f'rpn_obj={comp_sum.get("loss_objectness",0)/max(nb,1):.4f}  '
                  f'rpn_reg={comp_sum.get("loss_rpn_box_reg",0)/max(nb,1):.4f}  '
                  f'cls={comp_sum.get("loss_classifier",0)/max(nb,1):.4f}  '
                  f'reg={comp_sum.get("loss_box_reg",0)/max(nb,1):.4f}  '
                  f'lr_bb={lr_bb:.1e}  lr_hd={lr_hd:.1e}  '
                  f'[{time.time()-t0:.0f}s]')
    avg = max(nb,1)
    return run_loss/avg, {k:v/avg for k,v in comp_sum.items()}


@torch.no_grad()
def evaluate_map(model, loader, desc='val'):
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy',iou_type='bbox',
                                  class_metrics=True)
    for images,targets in loader:
        images = [x.to(device) for x in images]
        with torch.amp.autocast('cuda'):
            preds = model(images)
        metric.update([{k:v.cpu() for k,v in p.items()} for p in preds],
                      [{k:v.cpu() for k,v in t.items()} for t in targets])
    r = metric.compute()
    stats = {k: float(r[k]) for k in ['map','map_50','map_75','mar_100']}
    stats['map_small']  = float(r.get('map_small',-1))
    stats['map_medium'] = float(r.get('map_medium',-1))
    stats['map_large']  = float(r.get('map_large',-1))
    stats['map_per_class'] = r.get('map_per_class',torch.tensor([])).tolist()
    print(f'  [{desc}] mAP={stats["map"]:.4f}  AP50={stats["map_50"]:.4f}  '
          f'AP75={stats["map_75"]:.4f}  AP_s={stats["map_small"]:.4f}  '
          f'mAR={stats["mar_100"]:.4f}')
    return stats


In [32]:
@torch.no_grad()
def evaluate_classification(model, loader, iou_thr=0.50,
                            score_thr=0.05, split='val'):
    """IoU-matched classification accuracy on detected boxes."""
    model.eval()
    y_true, y_pred = [], []
    n_gt = n_matched = 0

    for images,targets in loader:
        images = [x.to(device) for x in images]
        with torch.amp.autocast('cuda'):
            preds = model(images)
        for pred,tgt in zip(preds,targets):
            gt_b  = tgt['boxes'].cpu()
            gt_l  = tgt['labels'].cpu()
            n_gt += len(gt_b)
            if len(gt_b)==0: continue
            keep  = pred['scores'].cpu()>=score_thr
            pb,pl = pred['boxes'].cpu()[keep], pred['labels'].cpu()[keep]
            if len(pb)==0: continue
            iou   = box_iou(gt_b, pb)
            matched = set()
            for gi in range(len(gt_b)):
                vals,idxs = iou[gi].sort(descending=True)
                for j,v in zip(idxs.tolist(),vals.tolist()):
                    if v<iou_thr or j in matched: continue
                    matched.add(j)
                    y_true.append(gt_l[gi].item())
                    y_pred.append(pl[j].item())
                    n_matched+=1; break

    acc = sum(a==b for a,b in zip(y_true,y_pred))/max(len(y_true),1)
    f1m = f1_score(y_true,y_pred,average='macro',
                   labels=list(range(1,N_CLS_WITH_BG)),
                   zero_division=0) if y_true else 0.0
    det_rec = n_matched/max(n_gt,1)

    print(f'  [{split}] cls_acc={acc:.4f}  macro_F1={f1m:.4f}  '
          f'det_recall={det_rec:.4f}  matched={n_matched}/{n_gt}')
    if y_true:
        print(classification_report(y_true,y_pred,
              labels=list(range(1,N_CLS_WITH_BG)),
              target_names=CLASS_NAMES,zero_division=0))
    return {'acc':acc,'f1m':f1m,'det_rec':det_rec,
            'n_gt':n_gt,'n_matched':n_matched,
            'y_true':y_true,'y_pred':y_pred}


@torch.no_grad()
def collect_all_predictions(model, loader, score_thr=0.01):
    model.eval()
    all_p,all_g=[],[]
    for images,targets in loader:
        images=[x.to(device) for x in images]
        with torch.amp.autocast('cuda'):
            preds=model(images)
        for pred,tgt in zip(preds,targets):
            keep=pred['scores'].cpu()>=score_thr
            all_p.append({'boxes':pred['boxes'].cpu()[keep],
                          'scores':pred['scores'].cpu()[keep],
                          'labels':pred['labels'].cpu()[keep]})
            all_g.append({'boxes':tgt['boxes'].cpu(),
                          'labels':tgt['labels'].cpu()})
    return all_p,all_g


def compute_per_class_curves(all_preds,all_gts,iou_thr=0.50,n_pts=101):
    thresholds=np.linspace(0,1,n_pts)
    curves={}
    for cls in range(1,N_CLS_WITH_BG):
        scores_all,tp_all,fp_all=[],[],[]
        n_gt_cls=0
        for pred,gt in zip(all_preds,all_gts):
            gm=gt['labels']==cls
            gb=gt['boxes'][gm]; n_gt_cls+=len(gb)
            pm=pred['labels']==cls
            if pm.sum()==0: continue
            pb,ps=pred['boxes'][pm],pred['scores'][pm]
            order=ps.argsort(descending=True); pb,ps=pb[order],ps[order]
            matched=torch.zeros(len(gb),dtype=torch.bool)
            for j in range(len(pb)):
                scores_all.append(ps[j].item())
                if len(gb)==0: tp_all.append(0);fp_all.append(1); continue
                ious=box_iou(pb[j:j+1],gb)[0]
                bi=ious.argmax().item()
                if ious[bi]>=iou_thr and not matched[bi]:
                    matched[bi]=True; tp_all.append(1); fp_all.append(0)
                else: tp_all.append(0); fp_all.append(1)
        if not scores_all or n_gt_cls==0: curves[cls]={'name':CLASS_NAMES[cls-1]}; continue
        idx=np.argsort(-np.array(scores_all))
        tp_c=np.cumsum(np.array(tp_all)[idx])
        fp_c=np.cumsum(np.array(fp_all)[idx])
        prec=tp_c/(tp_c+fp_c+1e-9); rec=tp_c/(n_gt_cls+1e-9)
        prec_t,rec_t,f1_t=[],[],[]
        for thr in thresholds:
            m=np.array(scores_all)[idx]>=thr
            tp_t=np.array(tp_all)[idx][m].sum()
            fp_t=np.array(fp_all)[idx][m].sum()
            fn_t=n_gt_cls-tp_t
            p=tp_t/(tp_t+fp_t+1e-9); r=tp_t/(tp_t+fn_t+1e-9)
            prec_t.append(p); rec_t.append(r)
            f1_t.append(2*p*r/(p+r+1e-9))
        ap=float(np.trapz(prec[::-1],rec[::-1]))
        curves[cls]={'name':CLASS_NAMES[cls-1],
                     'prec_curve':prec,'rec_curve':rec,
                     'prec_thr':np.array(prec_t),'rec_thr':np.array(rec_t),
                     'f1_thr':np.array(f1_t),'thresholds':thresholds,
                     'ap_aucpr':ap,'n_gt':n_gt_cls}
    return curves


In [33]:
def save_fig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    print(f'  saved → {Path(path).name}')


# ── FIGURE 1: training curves  (matches YOLO notebook 6-panel style)
def plot_training_curves(history):
    epochs   = history['epochs']
    map50    = history['val_map50']
    map5095  = history['val_map5095']
    best_idx = int(np.argmax(map50)) if map50 else 0
    best_ep  = epochs[best_idx]    if epochs else 0

    fig = plt.figure(figsize=(20,12))
    fig.suptitle(f'Swin-T + TransformerHead  —  Training Curves  [{TAG}]',
                 fontsize=14, y=1.01)
    gs = gridspec.GridSpec(2,3,figure=fig,hspace=0.42,wspace=0.35)

    # mAP@0.5
    ax=fig.add_subplot(gs[0,0])
    ax.plot(epochs,map50,'b-o',lw=2,ms=4,label='val mAP@0.5')
    if map50: ax.axhline(max(map50),color='green',ls='--',lw=1,
                         label=f'best={max(map50):.4f} (ep{best_ep})')
    ax.set_title('mAP@0.5'); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=.3)

    # mAP@0.5:0.95
    ax=fig.add_subplot(gs[0,1])
    ax.plot(epochs,map5095,'g-o',lw=2,ms=4,label='mAP@0.5:0.95')
    if map5095: ax.axhline(max(map5095),color='green',ls='--',lw=1,
                            label=f'best={max(map5095):.4f}')
    ax.set_title('mAP@0.5:0.95'); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=.3)

    # mAP@0.5 bar per epoch (gold = best)
    ax=fig.add_subplot(gs[0,2])
    colors=['gold' if i==best_idx else 'steelblue' for i in range(len(map50))]
    ax.bar(epochs,map50,color=colors,alpha=0.85)
    if map50: ax.axhline(max(map50),color='red',ls='--',lw=1.5,
                          label=f'best={max(map50):.4f}')
    ax.set_title('mAP@0.5 per epoch  (gold=best)')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=.3,axis='y')

    # epoch-over-epoch delta
    ax=fig.add_subplot(gs[1,0])
    if len(map50)>1:
        delta=[map50[i]-map50[i-1] for i in range(1,len(map50))]
        cols=['green' if d>=0 else 'red' for d in delta]
        ax.bar(epochs[1:],delta,color=cols,alpha=0.8)
        ax.axhline(0,color='black',lw=0.8)
    ax.set_title('Epoch-over-epoch Δ mAP@0.5')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Δ mAP'); ax.grid(alpha=.3,axis='y')

    # both metrics overlaid
    ax=fig.add_subplot(gs[1,1])
    ax.plot(epochs,map50,'b-',lw=2,label='mAP@0.5')
    ax.plot(epochs,map5095,'g-',lw=2,label='mAP@0.5:0.95')
    ax.fill_between(epochs,map5095,map50,alpha=0.12,color='blue')
    ax.set_title('mAP@0.5 vs mAP@0.5:0.95')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=.3)

    # summary table
    ax=fig.add_subplot(gs[1,2]); ax.axis('off')
    best_m5095=max(map5095) if map5095 else 0
    tbl=[['Metric','Value'],
         ['Architecture','Swin-T + TransformerHead'],
         ['Variation','End-to-End (all train)'],
         ['LR backbone',f'{LR_BACKBONE:.0e}'],
         ['LR head',f'{LR_HEAD:.0e}'],
         ['Best mAP@0.5',f'{max(map50):.4f}  (ep{best_ep})' if map50 else '-'],
         ['Best mAP@0.5:0.95',f'{best_m5095:.4f}' if map5095 else '-'],
         ['Image size',f'{IMG_MAX_SIZE}px'],
         ['Batch size',str(BATCH_SIZE)]]
    t=ax.table(cellText=tbl[1:],colLabels=tbl[0],loc='center',cellLoc='left')
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1,1.4)
    ax.set_title('Training summary',fontsize=10)

    fig.tight_layout()
    save_fig(fig, PLOT_DIR/'training_curves.png')


# ── FIGURE 2: per-class results on test set  (4-panel YOLO style)
def plot_per_class_results(test_metrics):
    ap50  = test_metrics.get('ap50',[])
    ap    = test_metrics.get('ap5095',[])
    prec  = test_metrics.get('prec',[])
    rec   = test_metrics.get('rec',[])
    if not ap50: return

    fig,axes=plt.subplots(2,2,figsize=(18,12))
    fig.suptitle(f'Swin-T + TransformerHead  —  Per-class Results (TEST set)  [{TAG}]',
                 fontsize=13,y=1.01)
    mean50=np.mean(ap50); mean_ap=np.mean(ap)

    for ax,vals,mean_v,xlabel,title,color in [
        (axes[0,0],ap50, mean50, 'AP@0.5',      'Per-class AP@0.5',      None),
        (axes[0,1],ap,   mean_ap,'AP@0.5:0.95', 'Per-class AP@0.5:0.95', None),
        (axes[1,0],prec, np.mean(prec) if prec else 0,'Precision','Per-class Precision','teal'),
        (axes[1,1],rec,  np.mean(rec)  if rec  else 0,'Recall',   'Per-class Recall',   'coral'),
    ]:
        if color is None:
            color=[('steelblue' if v>=mean_v else 'salmon') for v in vals]
        bars=ax.barh(CLASS_NAMES,vals,color=color,alpha=0.85)
        ax.axvline(mean_v,color='red',ls='--',lw=1.5,label=f'mean={mean_v:.4f}')
        ax.set_xlim(0,min(max(vals)+0.12,1.05) if vals else 1)
        ax.set_xlabel(xlabel); ax.set_title(title)
        ax.legend(fontsize=9); ax.grid(alpha=.3,axis='x')
        for bar,v in zip(bars,vals):
            ax.text(v+0.008,bar.get_y()+bar.get_height()/2,
                    f'{v:.3f}',va='center',fontsize=9)

    plt.tight_layout()
    save_fig(fig, PLOT_DIR/'test_per_class_results.png')


# ── FIGURE 3: loss curves  (4-panel YOLO style)
def plot_loss_curves(history):
    epochs=history['epochs']
    comp = {k[5:]: v for k,v in history.items() if k.startswith('comp_')}
    total=history.get('train_total',[])

    fig,axes=plt.subplots(2,2,figsize=(16,12))
    fig.suptitle(f'Swin-T + TransformerHead  —  Training Loss Curves  [{TAG}]',
                 fontsize=13,y=1.01)

    for ax,(key,label,color) in zip(
        [axes[0,0],axes[0,1],axes[1,0]],
        [('loss_classifier','ROI Classification loss','tomato'),
         ('loss_box_reg',   'ROI Box-reg loss',       'steelblue'),
         ('loss_objectness','RPN Objectness loss',     'mediumseagreen')]):
        vals=comp.get(key,[])
        ax.plot(epochs[:len(vals)],vals,f'-o',color=color,lw=2,ms=4)
        ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.grid(alpha=.3)
        if vals:
            ax.annotate(f'final={vals[-1]:.4f}',
                        xy=(epochs[len(vals)-1],vals[-1]),
                        xytext=(-40,15),textcoords='offset points',
                        fontsize=9,color=color,
                        arrowprops=dict(arrowstyle='->',color=color))

    # normalized all losses together
    ax=axes[1,1]
    for key,label,color in [
        ('loss_classifier','ROI cls','tomato'),
        ('loss_box_reg',   'ROI reg','steelblue'),
        ('loss_objectness','RPN obj','mediumseagreen'),
        ('loss_rpn_box_reg','RPN reg','orange')]:
        vals=comp.get(key,[])
        if vals and vals[0]!=0:
            norm=[v/vals[0] for v in vals]
            ax.plot(epochs[:len(norm)],norm,lw=2,label=label,color=color)
    ax.axhline(1.0,color='gray',ls='--',lw=0.8,alpha=0.5)
    ax.set_title('All losses normalized to epoch 1')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Relative loss')
    ax.legend(fontsize=8); ax.grid(alpha=.3)

    plt.tight_layout()
    save_fig(fig, PLOT_DIR/'loss_curves.png')


# ── FIGURE 4: mAP bar chart  (val vs test)
def plot_map_bar(det_results):
    keys  =['map','map_50','map_75','map_small','map_medium','map_large','mar_100']
    labels=['mAP','AP50','AP75','AP_s','AP_m','AP_l','mAR']
    splits=[s for s in ['val','test'] if s in det_results]
    x=np.arange(len(keys)); w=0.35
    fig,ax=plt.subplots(figsize=(12,5))
    for i,sp in enumerate(splits):
        vals=[max(det_results[sp].get(k,0),0) for k in keys]
        bars=ax.bar(x+i*w-w*len(splits)/4,vals,w,label=sp.upper(),alpha=0.85)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2,b.get_height()+.003,
                    f'{v:.3f}',ha='center',va='bottom',fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel('Score'); ax.set_title(f'COCO mAP — {TAG}')
    ax.legend(); ax.grid(axis='y',alpha=.3); ax.set_ylim(0,0.7)
    fig.tight_layout(); save_fig(fig, PLOT_DIR/'detection_map.png')


# ── FIGURE 5: per-class AP bar  (torchmetrics)
def plot_per_class_ap(det_results):
    for sp in ['val','test']:
        if sp not in det_results: continue
        aps=det_results[sp].get('map_per_class',[])
        if not aps: continue
        aps=list(aps)[:N_CLS]
        order=np.argsort(aps)
        fig,ax=plt.subplots(figsize=(8,5))
        bars=ax.barh([CLASS_NAMES[i] for i in order],
                     [aps[i] for i in order],color='steelblue',alpha=0.8)
        for b in bars:
            ax.text(b.get_width()+.002,b.get_y()+b.get_height()/2,
                    f'{b.get_width():.3f}',va='center',fontsize=8)
        ax.set_xlabel('AP'); ax.set_title(f'Per-class AP ({sp.upper()})')
        ax.set_xlim(0,max(aps+[.01])+.08); ax.grid(axis='x',alpha=.3)
        fig.tight_layout(); save_fig(fig, PLOT_DIR/f'per_class_ap_{sp}.png')


# ── FIGURE 6: confusion matrices
def plot_confusion_matrices(cls_data):
    for sp,cd in cls_data.items():
        yt,yp=cd.get('y_true',[]),cd.get('y_pred',[])
        if not yt: continue
        labels=list(range(1,N_CLS_WITH_BG))
        cm  =confusion_matrix(yt,yp,labels=labels)
        cm_n=cm.astype(float)/(cm.sum(axis=1,keepdims=True)+1e-9)
        fig,axes=plt.subplots(1,2,figsize=(16,6))
        for ax,mat,fmt,ttl in zip(axes,[cm,cm_n],['d','.2f'],
                                   ['Raw counts','Normalised']):
            sns.heatmap(mat,ax=ax,fmt=fmt,cmap='Blues',annot=True,
                        annot_kws={'size':7},linewidths=.3,
                        xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
            ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth')
            ax.set_title(f'Confusion matrix — {ttl} ({sp.upper()})')
        fig.tight_layout(); save_fig(fig, PLOT_DIR/f'confusion_matrix_{sp}.png')


# ── FIGURE 7: per-class classification accuracy on matched detections
def plot_cls_accuracy(cls_data):
    for sp,cd in cls_data.items():
        yt,yp=cd.get('y_true',[]),cd.get('y_pred',[])
        if not yt: continue
        acc_per=[]
        for cls in range(1,N_CLS_WITH_BG):
            gt_m=[a==cls for a in yt]
            if not any(gt_m): acc_per.append(0.0); continue
            acc_per.append(sum(a==b for a,b in zip(yt,yp) if a==cls)/sum(gt_m))
        order=np.argsort(acc_per)
        fig,ax=plt.subplots(figsize=(8,5))
        bars=ax.barh([CLASS_NAMES[i] for i in order],
                     [acc_per[i] for i in order],
                     color='mediumseagreen',alpha=0.8)
        for b in bars:
            ax.text(b.get_width()+.005,b.get_y()+b.get_height()/2,
                    f'{b.get_width():.3f}',va='center',fontsize=8)
        ax.set_xlabel('Accuracy'); ax.set_xlim(0,1.08)
        ax.set_title(f'Per-class accuracy on matched detections ({sp.upper()})')
        ax.grid(axis='x',alpha=.3)
        fig.tight_layout(); save_fig(fig, PLOT_DIR/f'cls_acc_matched_{sp}.png')


# ── per-class confidence curves helpers
def _plot_vs_conf(curves,split,key,ylabel,fname):
    fig,ax=plt.subplots(figsize=(9,7))
    colors=plt.cm.tab10(np.linspace(0,1,N_CLS))
    for cls in range(1,N_CLS_WITH_BG):
        c=curves.get(cls,{})
        if not c or key not in c: continue
        ax.plot(c['thresholds'],c[key],color=colors[cls-1],label=CLASS_NAMES[cls-1])
    ax.set_xlabel('Confidence threshold'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs confidence ({split.upper()})')
    ax.set_xlim(0,1); ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); save_fig(fig, PLOT_DIR/f'{fname}_{split}.png')

def plot_pr_curves(curves,split):
    fig,ax=plt.subplots(figsize=(9,7))
    colors=plt.cm.tab10(np.linspace(0,1,N_CLS))
    for cls in range(1,N_CLS_WITH_BG):
        c=curves.get(cls,{})
        if not c or 'rec_curve' not in c: continue
        ax.plot(c['rec_curve'],c['prec_curve'],
                color=colors[cls-1],
                label=f"{c['name']} ({c.get('ap_aucpr',0):.3f})")
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR curves per class ({split.upper()})')
    ax.set_xlim(0,1.02); ax.set_ylim(0,1.02)
    ax.legend(fontsize=8,loc='lower left'); ax.grid(alpha=.3)
    fig.tight_layout(); save_fig(fig, PLOT_DIR/f'pr_curve_per_class_{split}.png')

def plot_f1_confidence(c,sp):   _plot_vs_conf(c,sp,'f1_thr',  'F1',       'f1_confidence_per_class')
def plot_prec_confidence(c,sp): _plot_vs_conf(c,sp,'prec_thr','Precision','precision_confidence_per_class')
def plot_rec_confidence(c,sp):  _plot_vs_conf(c,sp,'rec_thr', 'Recall',   'recall_confidence_per_class')

def plot_ap_aucpr_bar(curves,split):
    aps={cls:curves[cls]['ap_aucpr'] for cls in curves if curves.get(cls,{}).get('ap_aucpr') is not None}
    if not aps: return 0.0,{}
    names=[CLASS_NAMES[c-1] for c in sorted(aps)]
    vals =[aps[c] for c in sorted(aps)]
    mean_ap=float(np.mean(vals))
    order=np.argsort(vals)
    fig,ax=plt.subplots(figsize=(8,5))
    bars=ax.barh([names[i] for i in order],[vals[i] for i in order],
                 color='mediumpurple',alpha=0.8)
    for b in bars:
        ax.text(b.get_width()+.003,b.get_y()+b.get_height()/2,
                f'{b.get_width():.3f}',va='center',fontsize=8)
    ax.axvline(mean_ap,color='tomato',ls='--',label=f'Mean={mean_ap:.3f}')
    ax.set_xlabel('AP (AUC-PR)'); ax.set_xlim(0,max(vals)+.1)
    ax.set_title(f'Per-class AP from AUC-PR ({split.upper()})')
    ax.legend(fontsize=9); ax.grid(axis='x',alpha=.3)
    fig.tight_layout(); save_fig(fig, PLOT_DIR/f'ap_aucpr_per_class_{split}.png')
    return mean_ap, aps


# ── FIGURE 8: sample detections GT vs Predicted
def gt_vs_model(model_name, ckpt_path, loader, n_samples=8, score_thr=0.40):
    palette=[(255,56,56),(255,157,151),(255,112,31),(255,178,29),
             (207,210,49),(72,249,10),(146,204,23),(61,219,134),
             (32,177,255),(0,226,252)]
    m=build_model(pretrained=False).to(device)
    ck=torch.load(ckpt_path,map_location=device)
    m.load_state_dict(ck['model_state_dict']); m.eval()

    dataset=loader.dataset
    idxs=np.random.choice(len(dataset),min(n_samples,len(dataset)),replace=False)
    fig,axes=plt.subplots(n_samples,2,figsize=(14,n_samples*3.5))
    if n_samples==1: axes=[axes]
    fig.suptitle(f'GT (left) vs Predicted (right)  —  {model_name}',fontsize=12)

    with torch.no_grad():
        for row,idx in enumerate(idxs):
            img_t,tgt=dataset[idx]
            with torch.amp.autocast('cuda'):
                pred=m([img_t.to(device)])[0]
            img_np=(img_t.permute(1,2,0).numpy()*255).astype(np.uint8)
            for col,(boxes,labels,scores,is_pred) in enumerate([
                (tgt['boxes'].tolist(),tgt['labels'].tolist(),None,False),
                (pred['boxes'].cpu().tolist(),pred['labels'].cpu().tolist(),
                 pred['scores'].cpu().tolist(),True)]):
                im=Image.fromarray(img_np.copy()); dr=ImageDraw.Draw(im)
                n_shown=0
                for i,(b,l) in enumerate(zip(boxes,labels)):
                    if is_pred and scores[i]<score_thr: continue
                    c=palette[(l-1)%10] if 0<l<=N_CLS else (255,255,255)
                    dr.rectangle(b,outline=c,width=2)
                    nm=CLASS_NAMES[l-1] if 0<l<=N_CLS else str(l)
                    lbl=nm if not is_pred else f'{nm} {scores[i]:.2f}'
                    dr.text((b[0],max(b[1]-11,0)),lbl,fill=c)
                    n_shown+=1
                axes[row][col].imshow(np.array(im)); axes[row][col].axis('off')
                ttl=f'GT (n={n_shown})' if not is_pred else f'Pred (n={n_shown}, thr={score_thr})'
                axes[row][col].set_title(ttl,fontsize=9)
    plt.tight_layout()
    save_fig(fig, PLOT_DIR/'sample_detections.png')
    print(f'Saved sample detections for {model_name}')


In [34]:
history    = defaultdict(list)
best_map50 = 0.0
best_epoch = 0

for epoch in range(1, EPOCHS+1):
    t0=time.time()
    print(f"\n{'='*60}")
    print(f'Epoch {epoch}/{EPOCHS}  '
          f'lr_bb={optimizer.param_groups[0]["lr"]:.2e}  '
          f'lr_hd={optimizer.param_groups[1]["lr"]:.2e}')
    print(f"{'='*60}")

    avg_loss, comp = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, epoch)

    history['epochs'].append(epoch)
    history['train_total'].append(avg_loss)
    for k,v in comp.items():
      history[f'comp_{k}'].append(v)
    history['lr_backbone'].append(optimizer.param_groups[0]['lr'])
    history['lr_head'].append(optimizer.param_groups[1]['lr'])

    print('  evaluating val...')
    val_res = evaluate_map(model, val_loader, 'val')
    m50     = val_res.get('map_50', 0.0)
    m5095   = val_res.get('map',    0.0)
    history['val_map50'].append(m50)
    history['val_map5095'].append(m5095)
    history['val_map75'].append(val_res.get('map_75',0.0))
    history['val_mar100'].append(val_res.get('mar_100',0.0))

    elapsed=time.time()-t0
    print(f'  loss={avg_loss:.4f}  mAP@0.5={m50:.4f}  '
          f'mAP@0.5:0.95={m5095:.4f}  time={elapsed:.0f}s')

    ckpt={'epoch':epoch,
          'model_state_dict':model.state_dict(),
          'optimizer_state_dict':optimizer.state_dict(),
          'map_50':m50}
    torch.save(ckpt, CKPT_DIR/f'swint_{TAG}_ep{epoch:03d}.pt')
    if m50>best_map50:
        best_map50=m50; best_epoch=epoch
        torch.save(ckpt, CKPT_DIR/f'swint_{TAG}_best.pt')
        print(f'  >> new best mAP@0.5={best_map50:.4f}')

print(f'\nTraining complete.')
print(f'Best mAP@0.5={best_map50:.4f} at epoch {best_epoch}')
BEST_PT = CKPT_DIR / f'swint_{TAG}_best.pt'



Epoch 1/30  lr_bb=2.50e-07  lr_hd=2.50e-06


/tmp/ipykernel_2231/2696771963.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  [1][0000/0269]  loss=3.9390  rpn_obj=0.6990  rpn_reg=1.0303  cls=1.5244  reg=0.6853  lr_bb=5.0e-07  lr_hd=5.0e-06  [2s]
  [1][0050/0269]  loss=2.3213  rpn_obj=0.5247  rpn_reg=0.5392  cls=0.7326  reg=0.5247  lr_bb=1.3e-05  lr_hd=1.3e-04  [36s]
  [1][0100/0269]  loss=2.1358  rpn_obj=0.3782  rpn_reg=0.4474  cls=0.7050  reg=0.6052  lr_bb=2.6e-05  lr_hd=2.6e-04  [70s]
  [1][0150/0269]  loss=2.0339  rpn_obj=0.3131  rpn_reg=0.4041  cls=0.6795  reg=0.6372  lr_bb=3.8e-05  lr_hd=3.8e-04  [103s]
  [1][0200/0269]  loss=1.9689  rpn_obj=0.2774  rpn_reg=0.3774  cls=0.6613  reg=0.6529  lr_bb=5.0e-05  lr_hd=5.0e-04  [137s]
  [1][0250/0269]  loss=1.9067  rpn_obj=0.2510  rpn_reg=0.3609  cls=0.6422  reg=0.6527  lr_bb=5.0e-05  lr_hd=5.0e-04  [171s]
  evaluating val...


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


  [val] mAP=0.0495  AP50=0.1316  AP75=0.0277  AP_s=0.0369  mAR=0.1191
  loss=1.8878  mAP@0.5=0.1316  mAP@0.5:0.95=0.0495  time=216s
  >> new best mAP@0.5=0.1316

Epoch 2/30  lr_bb=5.00e-05  lr_hd=5.00e-04
  [2][0000/0269]  loss=1.6816  rpn_obj=0.1454  rpn_reg=0.2993  cls=0.5951  reg=0.6417  lr_bb=5.0e-05  lr_hd=5.0e-04  [2s]
  [2][0050/0269]  loss=1.5771  rpn_obj=0.1345  rpn_reg=0.2816  cls=0.5332  reg=0.6277  lr_bb=5.0e-05  lr_hd=5.0e-04  [35s]
  [2][0100/0269]  loss=1.5496  rpn_obj=0.1304  rpn_reg=0.2742  cls=0.5249  reg=0.6200  lr_bb=5.0e-05  lr_hd=5.0e-04  [69s]
  [2][0150/0269]  loss=1.5300  rpn_obj=0.1286  rpn_reg=0.2705  cls=0.5183  reg=0.6126  lr_bb=5.0e-05  lr_hd=5.0e-04  [102s]
  [2][0200/0269]  loss=1.5193  rpn_obj=0.1272  rpn_reg=0.2686  cls=0.5146  reg=0.6089  lr_bb=5.0e-05  lr_hd=5.0e-04  [136s]
  [2][0250/0269]  loss=1.5042  rpn_obj=0.1257  rpn_reg=0.2673  cls=0.5086  reg=0.6025  lr_bb=5.0e-05  lr_hd=5.0e-04  [169s]
  evaluating val...
  [val] mAP=0.0875  AP50=0.2105  AP

In [35]:
print(f'\nLoading best checkpoint: {BEST_PT}')
ck=torch.load(BEST_PT,map_location=device)
model.load_state_dict(ck['model_state_dict'])
model.eval()

print('\n── Detection mAP ──────────────────────────────────────────')
det_results={}
for sp,loader in [('val',val_loader),('test',test_loader)]:
    print(f'\n  {sp.upper()}')
    det_results[sp]=evaluate_map(model,loader,desc=sp)

print('\n── Classification on matched detections ───────────────────')
cls_data={}
for sp,loader in [('val',val_loader),('test',test_loader)]:
    print(f'\n  {sp.upper()}')
    cls_data[sp]=evaluate_classification(model,loader,split=sp)



Loading best checkpoint: /content/drive/MyDrive/VisDrone/checkpoints/swint_opt3_e2e/swint_swint_opt3_e2e_best.pt

── Detection mAP ──────────────────────────────────────────

  VAL
  [val] mAP=0.2093  AP50=0.3968  AP75=0.1907  AP_s=0.1517  mAR=0.3296

  TEST
  [test] mAP=0.1667  AP50=0.3251  AP75=0.1528  AP_s=0.1157  mAR=0.2842

── Classification on matched detections ───────────────────

  VAL
  [val] cls_acc=0.7759  macro_F1=0.5979  det_recall=0.7189  matched=27863/38759
                 precision    recall  f1-score   support

     pedestrian       0.81      0.86      0.83      5251
         people       0.74      0.68      0.71      3114
        bicycle       0.41      0.50      0.45       630
            car       0.91      0.88      0.90     12043
            van       0.40      0.51      0.45      1675
          truck       0.48      0.52      0.50       552
       tricycle       0.46      0.43      0.45       741
awning-tricycle       0.30      0.34      0.32       347
       

In [36]:
print('\nGenerating plots...')

# ── Training + loss curves
plot_training_curves(history)
plot_loss_curves(history)

# ── Detection mAP bar  +  per-class AP
plot_map_bar(det_results)
plot_per_class_ap(det_results)

# ── Classification: confusion matrices + per-class accuracy
plot_confusion_matrices(cls_data)
plot_cls_accuracy(cls_data)

# ── Per-class test results  (AP@0.5, AP@0.5:0.95, Precision, Recall)
test_m = det_results.get('test',{})
per_cls_ap50   = test_m.get('map_per_class',[0]*N_CLS)
# precision/recall per class from classification eval
def _per_cls_scores(cd, fn):
    yt,yp=cd.get('y_true',[]),cd.get('y_pred',[])
    if not yt: return [0]*N_CLS
    return [fn(yt,yp,labels=[c],average='macro',zero_division=0)
            for c in range(1,N_CLS_WITH_BG)]
prec_per = _per_cls_scores(cls_data.get('test',{}), precision_score)
rec_per  = _per_cls_scores(cls_data.get('test',{}), recall_score)
test_metrics_plot = {
    'ap50':   [float(x) for x in per_cls_ap50[:N_CLS]],
    'ap5095': [0]*N_CLS,   # torchmetrics returns mean only; use 0 placeholder
    'prec':   prec_per,
    'rec':    rec_per,
}
plot_per_class_results(test_metrics_plot)

# ── Sample detections
gt_vs_model('Swin-T E2E', str(BEST_PT), val_loader, n_samples=8)

# ── Per-class confidence curves (val + test)
per_class_curves={}
for sp,loader in [('val',val_loader),('test',test_loader)]:
    print(f'\n  Collecting predictions [{sp}]...')
    all_p,all_g=collect_all_predictions(model,loader)
    curves=compute_per_class_curves(all_p,all_g)
    per_class_curves[sp]=curves
    plot_pr_curves(curves,sp)
    plot_f1_confidence(curves,sp)
    plot_prec_confidence(curves,sp)
    plot_rec_confidence(curves,sp)
    mean_ap,_=plot_ap_aucpr_bar(curves,sp)
    print(f'  [{sp}] Mean AUC-PR = {mean_ap:.4f}')

print('\nAll plots saved to', PLOT_DIR)



Generating plots...


/tmp/ipykernel_2231/1343758757.py:79: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  saved → training_curves.png
  saved → loss_curves.png
  saved → detection_map.png
  saved → per_class_ap_val.png
  saved → per_class_ap_test.png
  saved → confusion_matrix_val.png
  saved → confusion_matrix_test.png
  saved → cls_acc_matched_val.png
  saved → cls_acc_matched_test.png
  saved → test_per_class_results.png
  Swin-T body    : 27.52M  [lr=5e-05]
  FPN            : 2.73M  [lr=5e-04]
  RPN            : 0.59M  [lr=5e-04]
  TransformerHead: 554K  [lr=5e-04]
  Total          : 31.40M  (all trainable)
  saved → sample_detections.png
Saved sample detections for Swin-T E2E



/tmp/ipykernel_2231/2842922396.py:103: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ap=float(np.trapz(prec[::-1],rec[::-1]))


  saved → pr_curve_per_class_val.png
  saved → f1_confidence_per_class_val.png
  saved → precision_confidence_per_class_val.png
  saved → recall_confidence_per_class_val.png


/tmp/ipykernel_2231/1343758757.py:293: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout(); save_fig(fig, PLOT_DIR/f'ap_aucpr_per_class_{split}.png')


  saved → ap_aucpr_per_class_val.png
  [val] Mean AUC-PR = -0.3934

  saved → pr_curve_per_class_test.png
  saved → f1_confidence_per_class_test.png
  saved → precision_confidence_per_class_test.png
  saved → recall_confidence_per_class_test.png
  saved → ap_aucpr_per_class_test.png
  [test] Mean AUC-PR = -0.3214

All plots saved to /content/drive/MyDrive/VisDrone/plots/swint_opt3_e2e


In [37]:
def t2s(v):
    if isinstance(v,torch.Tensor):
        return v.item() if v.numel()==1 else v.tolist()
    if isinstance(v,np.ndarray): return v.tolist()
    return v

sd={}
for sp in ['val','test']:
    if sp not in det_results: continue
    r=det_results[sp]; cd=cls_data.get(sp,{}); pc=per_class_curves.get(sp,{})
    ap_pr={CLASS_NAMES[c-1]:round(pc[c]['ap_aucpr'],4)
           for c in pc if pc.get(c,{}).get('ap_aucpr') is not None}
    sd[sp]={
        'detection':{k:t2s(r.get(k,0)) for k in
            ['map','map_50','map_75','map_small','map_medium',
             'map_large','mar_100','map_per_class']},
        'classification':{
            'accuracy':   round(float(cd.get('acc',0)),4),
            'macro_f1':   round(float(cd.get('f1m',0)),4),
            'det_recall': round(float(cd.get('det_rec',0)),4),
            'n_gt':       int(cd.get('n_gt',0)),
            'n_matched':  int(cd.get('n_matched',0)),
        },
        'ap_aucpr_per_class': ap_pr,
        'mean_ap_aucpr': round(float(np.mean(list(ap_pr.values()))) if ap_pr else 0,4),
    }

sd['training']={
    'tag':TAG, 'mode':'end_to_end',
    'best_epoch':best_epoch, 'best_map50':best_map50,
    'epochs':history.get('epochs',[]),
    'val_map50'   :[round(x,5) for x in history.get('val_map50',[])],
    'val_map5095' :[round(x,5) for x in history.get('val_map5095',[])],
    'train_total' :[round(x,5) for x in history.get('train_total',[])],
}
sd['config']={
    'model':          'Swin-T + Faster R-CNN + TransformerHead (Option 3)',
    'head_type':      'transformer_opt3',
    'variation':      'end_to_end',
    'trainable_desc': 'All params (Swin body lr=5e-5, rest lr=5e-4)',
    'img_max_size':   IMG_MAX_SIZE,
    'n_epochs':       EPOCHS,
    'lr_backbone':    LR_BACKBONE,
    'lr_head':        LR_HEAD,
    'weight_decay':   WEIGHT_DECAY,
    'batch_size':     BATCH_SIZE,
    'best_epoch':     best_epoch,
    'best_map50':     best_map50,
}

json_path=PLOT_DIR/f'metrics_{TAG}.json'
with open(json_path,'w') as f:
    json.dump(sd,f,indent=2,default=t2s)
print(f'Metrics saved → {json_path}')

# ── Printed summary  (matches ResNet format)
print(f"\n{'='*60}")
print(f'SUMMARY: Swin-T End-to-End, {IMG_MAX_SIZE}px')
print(f"{'='*60}")
print(f"{'Metric':<25s} {'Val':>10s} {'Test':>10s}")
print('-'*45)
for lab,k in [('mAP@0.5:0.95','map'),('mAP@0.5','map_50'),
               ('mAP@0.75','map_75'),('mAP small','map_small'),
               ('mAR@100','mar_100')]:
    vs=[]
    for sp in ['val','test']:
        v=det_results.get(sp,{}).get(k,0)
        v=v.item() if isinstance(v,torch.Tensor) and v.numel()==1 else v
        vs.append(max(float(v),0))
    print(f'  {lab:<23s} {vs[0]:>10.4f} {vs[1]:>10.4f}')
print()
for sp in ['val','test']:
    cd=cls_data.get(sp,{})
    m=sd.get(sp,{})
    print(f'  [{sp.upper()}]')
    print(f'    Cls accuracy : {cd.get("acc",0):.4f}')
    print(f'    Macro F1     : {cd.get("f1m",0):.4f}')
    print(f'    Det recall   : {cd.get("det_rec",0):.4f}')
    print(f'    Mean AUC-PR  : {m.get("mean_ap_aucpr",0):.4f}')
print(f"\n  Best epoch: {best_epoch}  mAP@0.5={best_map50:.4f}")
print(f"{'='*60}")


Metrics saved → /content/drive/MyDrive/VisDrone/plots/swint_opt3_e2e/metrics_swint_opt3_e2e.json

SUMMARY: Swin-T End-to-End, 1024px
Metric                           Val       Test
---------------------------------------------
  mAP@0.5:0.95                0.2093     0.1667
  mAP@0.5                     0.3968     0.3251
  mAP@0.75                    0.1907     0.1528
  mAP small                   0.1517     0.1157
  mAR@100                     0.3296     0.2842

  [VAL]
    Cls accuracy : 0.7759
    Macro F1     : 0.5979
    Det recall   : 0.7189
    Mean AUC-PR  : -0.3934
  [TEST]
    Cls accuracy : 0.7739
    Macro F1     : 0.5882
    Det recall   : 0.6441
    Mean AUC-PR  : -0.3214

  Best epoch: 29  mAP@0.5=0.3968
